# NextPyper manuscript figures

### Method comparison plotlines

The figure designed for visualising the comparative success of the three selected methods on the simulated data. The figure is meant to be complete and should end up in the supplementary material. A version using stacked bar plot for selected categories shall be used in the body of the manuscript.
This figure depicts the mean and standard variation for selected features across plodies, that is the statistics are derived from all accessions at a given ploidy level. We investigated the methods' performance across various tree depths, that is the influence of the probe/target similarity and several read coverage that would affect the assembly graph construction.

The file structure must consist of a directory (e.g. 'Myears_10_0') that contains all results at a given tree depth. Within it, folders for each coverage (e.g. 'cov_6_allele1') should contain separated results for each method ('Myears_10_0_cov_6_allele1_captus.tsv', 'Myears_10_0_cov_6_allele1_hybpiper.tsv', 'Myears_10_0_cov_6_allele1_nextpyper.tsv').

In [6]:
from pathlib import Path
from dataclasses import dataclass, field
from collections import namedtuple
from statistics import mean, stdev
from typing import Literal

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Headers of the result file
headers=['taxon', 'unique', 'duplicated', 'fragmented', 'missing', 'chimeras_true', 'chimeras_fail']

def get_stdev(lst:list) -> float:
    """
    Compute the standard deviation of a list of numbers. Returns 0 if the list has a single element, otherwise returns the standard deviation.
    """
    if len(lst) == 1:
        return 0
    else:
        return stdev(lst)

ploidy_dict = {'Deca':10, 'Di':2, 'Dodeca':12, 'Hexa':6, 'Octa':8, 'Tetra':4}

def create_table(path:Path)->pd.DataFrame:
    """Fuse the three tables in a new dataframe.
    Sum up the two chimeric categories.
    Add ploidy information from the ploidy_dict."""
    frames = []
    #fuse dataframes, compute chimeras
    for tsv in path.glob('*.tsv'):
        method=tsv.name.split('_')[-1].replace('.tsv','')
        df = pd.read_table(tsv,header=None,names=headers)
        df["chimeras"] = df['chimeras_true'] + df['chimeras_fail']
        df['method'] = method
        frames.append(df)
    concat = pd.concat(frames)
    new_data = []
    for row in concat.itertuples():
        new_row = [row.taxon, row.method, row.unique, row.duplicated, row.fragmented, row.missing, row.chimeras,
                   ploidy_dict[row.taxon.split('_')[0]]]
        new_data.append(new_row)

    return pd.DataFrame(sorted(new_data, key=lambda x:(x[-1], x[0], x[1])), columns=['taxon', 'method', 'unique', 'duplicated', 'fragmented', 'missing', 'chimeras','ploidy'])

def normalize_data(df: pd.DataFrame, probe_number:int) -> pd.DataFrame:
    """Normalize the data across samples to get recovery percentage, get mean and stdev"""
       #compute mean and stdev
    new_data = []
    current_ploidy = None
    method =  None
    unique = []
    duplicated = []
    fragmented = []
    missing = []
    chimeras = []
    sorted_rows = sorted([x for x in df.itertuples()], key=lambda x: (x.method, x.ploidy,))

    for row in sorted_rows:
        ploidy = row.ploidy
        if current_ploidy is None:
            current_ploidy = ploidy
            method = row.method
        if ploidy != current_ploidy:
            new_row = [current_ploidy,method,mean(unique),get_stdev(unique), mean(duplicated),
                       get_stdev(duplicated),mean(fragmented),get_stdev(fragmented),mean(missing),get_stdev(missing),mean(chimeras),get_stdev(chimeras),]
            new_data.append(new_row)
            current_ploidy = ploidy
            method = row.method
            # below the 0.5 value is meant to get the expected number of sequences given the ploidy, i.e. a diploid can only contain a single set of probes.
            unique = [(row.unique / (probe_number*0.5*ploidy))*100]
            duplicated = [(row.duplicated/ (probe_number*0.5*ploidy))*100]
            fragmented = [(row.fragmented/ (probe_number*0.5*ploidy))*100]
            missing = [(row.missing/ (probe_number*0.5*ploidy))*100]
            chimeras = [(row.chimeras/ (probe_number*0.5*ploidy))*100]
        else:
            unique.append((row.unique/ (probe_number*0.5*ploidy))*100)
            duplicated.append((row.duplicated/ (probe_number*0.5*ploidy))*100)
            fragmented.append((row.fragmented/ (probe_number*0.5*ploidy))*100)
            missing.append((row.missing/ (probe_number*0.5*ploidy))*100)
            chimeras.append((row.chimeras/ (probe_number*0.5*ploidy))*100)
    new_row = [current_ploidy,method,mean(unique),get_stdev(unique), mean(duplicated),
               get_stdev(duplicated),mean(fragmented),get_stdev(fragmented),mean(missing),get_stdev(missing),mean(chimeras),get_stdev(chimeras),]
    new_data.append(new_row)
    final_df = pd.DataFrame(new_data, columns=['ploidy', 'method', 'unique_mean', 'unique_stdev', 'duplicated_mean', 'duplicated_stdev',
                                               'fragmented_mean', 'fragmented_stdev', 'missing_mean', 'missing_stdev',
                                                'chimeras_mean', 'chimeras_stdev'])
    return final_df

In [7]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

target_year = Path('/home/yjkbertrand/Documents/projects/fumaria_sim/results/Myears_10_0')

years = str(target_year.name).replace('Myears_', "").replace('_','.')
variable_names = ['unique', 'duplicated', 'fragmented', 'chimeras', 'missing']
coverages = [6,10,20,30,50,100]

total_rows = len(coverages)
total_cols = len(variable_names)

fig = make_subplots(rows=total_rows, cols=total_cols, shared_xaxes=True, vertical_spacing=0.02,subplot_titles=variable_names,
                    shared_yaxes=True)

# Adjust layout
fig.update_layout(height=2000, width=1500, title_text=f"Tree depth:{years} M years",
    template="plotly",
    yaxis_range=[0,100], xaxis=dict(
    tickmode = 'array',
    tickvals = [2, 4, 6, 8, 10, 12],
    ticktext = ['0', '4', '6', '8', '10', '12']),
    yaxis=dict(
        title=dict(
            text="% recovered sequences"
        )),
    legend=dict(
            title=dict(
                text="Method"
            )
        ),
    )

# Loop over coverage (rows)
row_col_comb = []
for idx_0, cov in enumerate(coverages):

    row = idx_0 +1
    target = target_year / f"cov_{cov}_allele1"
    # Create df for each allele
    df = create_table(target)
    normal=normalize_data(df, 333)

    # Loop over statistics (columns)
    for idx_1, (mean_val, stdev_val) in enumerate([(f'{var}_mean', f'{var}_stdev') for var in variable_names]):
        col = idx_1 + 1
        legend=True if (idx_0, idx_1) == (0,0) else False
        row_col_comb.append((row, col))
        fig.add_trace(go.Scatter(x=normal[normal['method'] == 'captus']['ploidy'],
        y=normal[normal['method'] == 'captus'][mean_val],
        name='captus', showlegend=legend,
        marker_color='lightsalmon',
        error_y=dict(
        type='data',
        array=normal[normal['method'] == 'captus'][stdev_val],
        visible=True)),
        row=row, col=col)

        fig.add_trace(go.Scatter( x=normal[normal['method'] == 'nextpyper']['ploidy'],
        y=normal[normal['method'] == 'nextpyper'][mean_val],
        name='nextpyper', showlegend=legend,
        marker_color="goldenrod",
        error_y=dict(
        type='data',
        array=normal[normal['method'] == 'nextpyper'][stdev_val],
        visible=True)),
        row=row, col=col)

        fig.add_trace(go.Scatter( x=normal[normal['method'] == 'hybpiper']['ploidy'],
        y=normal[normal['method'] == 'hybpiper'][mean_val],
        name='hybpiper', showlegend=legend,
        marker_color='indianred',
        error_y=dict(
        type='data',
        array=normal[normal['method'] == 'hybpiper'][stdev_val],
        visible=True)),
        row=row, col=col)

# Adjust axes
for (row, col) in row_col_comb:
    if row == len(coverages):
        fig.update_xaxes(title_text="ploidy levels", row=row, col=col, tickmode = 'array', tickvals = [2, 4, 6, 8, 10, 12],
            ticktext = ['2', '4', '6', '8', '10', '12'])
    else:
        fig.update_xaxes(row=row, col=col, tickmode = 'array', tickvals = [2, 4, 6, 8, 10, 12],
            ticktext = ['2', '4', '6', '8', '10', '12'])
    if col == 1:
        fig.update_yaxes(title_text="% recovered sequences", range=[0,100],row=row, col=col)
    elif col == len(variable_names):
        depth = coverages[row-1]
        fig.update_yaxes(title_text=f"coverage {depth}", range=[0,100],row=row, col=col,side="right",)

fig.show()
